# Which pages should an editor review first?

**An honest content-decline ranking study across 120,507 pages**

*Muneeb Ur Rehman · Refresh / Content Opportunity Scoring · August 2026*

## Abstract

FlyRank's Refresh / Content Opportunity Scoring lane addresses a practical content-operations problem: when search performance declines across more pages than a content team can inspect, which pages should an editor review first? From FlyRank's pseudonymized warehouse release of 78,835,655 daily performance rows, I aggregated selected January–March 2026 partitions into a 120,507-page March analysis frame while excluding identifiers and signals unavailable before the outcome window. A random forest used four prior-window features—impressions, clicks, average position, and active days—and was evaluated with leave-one-client-out Precision@50 after label, timing, population, and positive-control leakage checks. Across 29 scoreable held-out clients, mean Precision@50 was 0.370, the model beat each client's own base rate in 22 cases, and mean per-client lift was 1.26×, although client results ranged from 0.020 to 0.760. The result is a decision-support case study for the lane: a human-reviewed prioritization queue with reason codes and monitoring gates, not an automated editing system, a causal recovery claim, or a guarantee for new or small clients.

> **Scope in one line:** the release contains about 78.8M daily rows; the model was evaluated on a 120,458-row complete-case subset of a 120,507-page aggregate—not on 79M independent training examples.


## 1. Introduction / problem statement

FlyRank's Refresh / Content Opportunity Scoring lane starts from a practical content-operations problem: search-performance monitoring can surface more possible declines than a content team can investigate. This project turns that problem into a public-safe decision-support case study by asking: **which pages should enter a limited human review queue first?** One row is one pseudonymized content item; the model produces a score used for ranking, and an editor decides whether to diagnose, defer, monitor, or propose an edit.

Precision@50 is the primary metric because reviewer capacity is the constraint. A false positive consumes recoverable review time; a false negative may leave a real decline unseen until the next review cycle. The model therefore needs to concentrate observed declines near the top, but it must not be allowed to diagnose a cause or publish a change.

The original hypothesis was that a learned ranker could outperform a transparent “stale and visible” rule. The audit changed that story: the freshness input was not available at the analysis boundary for most rows, so the final defensible baseline is random ordering at each held-out client's observed decline rate. The old rule is retained only as an audit-trail result, not as the headline comparator.


In [1]:
import hashlib
import json
import os
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 140)


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "work/notebooks/w06_validation_audit.ipynb").exists() and (candidate / "README.md").exists():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook from inside the flyrank-ml-track repository.")


PROJECT_ROOT = find_project_root()
W06_PATH = PROJECT_ROOT / "work/notebooks/w06_validation_audit.ipynb"
W07_PATH = PROJECT_ROOT / "work/outputs/w07_playbook_metrics.json"
FIGURE_DIR = PROJECT_ROOT / "work/figures"
DOC_ASSET_DIR = PROJECT_ROOT / "docs/assets"
OUTPUT_DIR = PROJECT_ROOT / "work/outputs"
for directory in (FIGURE_DIR, DOC_ASSET_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

w06 = json.loads(W06_PATH.read_text(encoding="utf-8"))
w07 = json.loads(W07_PATH.read_text(encoding="utf-8"))


def all_saved_output_text(notebook):
    chunks = []
    for cell in notebook.get("cells", []):
        for output in cell.get("outputs", []):
            if "text" in output:
                chunks.append("".join(output["text"]))
            data = output.get("data", {})
            if "text/plain" in data:
                chunks.append("".join(data["text/plain"]))
    return "\n".join(chunks)


w06_text = all_saved_output_text(w06)
required_receipts = [
    "Honest LOCO over 29 scoreable clients (of 42 total)",
    "mean P@50   0.370",
    "beat their own base rate: 22 of 29 clients",
    "mean lift over each client's own base rate: 1.26x",
    "Top-50 queue: 28 of 50 really declined",
    "days_since_update negative: 65,211 of 81,446 rows (80.1%)",
]
missing = [receipt for receipt in required_receipts if receipt not in w06_text]
assert not missing, f"Week-6 saved outputs are missing required receipts: {missing}"

# Parse only aggregates from the saved public-safe LOCO table; client identifiers are discarded.
loco_block = next(
    "\n".join(
        "".join(output.get("text", []))
        for output in cell.get("outputs", [])
        if "text" in output
    )
    for cell in w06["cells"]
    if "Honest LOCO over 29 scoreable clients" in "\n".join(
        "".join(output.get("text", [])) for output in cell.get("outputs", []) if "text" in output
    )
)
row_pattern = re.compile(r"^client_\S+[ \t]+(\d+)[ \t]+([0-9.]+)[ \t]+([0-9.]+|NaN)(?:[ \t]+.*)?$", re.MULTILINE)
parsed_rows = [
    {"n": int(n), "base_rate": float(base), "p50": None if p50 == "NaN" else float(p50)}
    for n, base, p50 in row_pattern.findall(loco_block)
]
scoreable = pd.DataFrame([row for row in parsed_rows if row["p50"] is not None])
assert len(parsed_rows) == 42 and len(scoreable) == 29

validation = w07["validation_receipt"]
assert validation["loco_scoreable_clients"] == 29
assert np.isclose(validation["loco_mean_precision_at_50"], scoreable["p50"].mean(), atol=0.001)

FINAL = {
    "extracted_pages": 120_507,
    "honest_complete_rows": 120_458,
    "honest_total_clients": 42,
    "scoreable_clients": int(len(scoreable)),
    "unscoreable_clients": 13,
    "model_mean_p50": float(scoreable["p50"].mean()),
    "model_median_p50": float(scoreable["p50"].median()),
    "model_min_p50": float(scoreable["p50"].min()),
    "model_max_p50": float(scoreable["p50"].max()),
    "mean_client_base_rate": float(scoreable["base_rate"].mean()),
    "clients_beating_base": int((scoreable["p50"] > scoreable["base_rate"]).sum()),
    "mean_per_client_lift": float((scoreable["p50"] / scoreable["base_rate"]).mean()),
}
print("Evidence receipts matched Week 6 and Week 7.")
print(f"Final frame: {FINAL['honest_complete_rows']:,} rows | {FINAL['honest_total_clients']} clients | {FINAL['scoreable_clients']} scoreable")
print(f"Primary result: mean P@50 {FINAL['model_mean_p50']:.3f} | mean client base rate {FINAL['mean_client_base_rate']:.3f}")
print(f"Wins: {FINAL['clients_beating_base']} of {FINAL['scoreable_clients']} | mean per-client lift {FINAL['mean_per_client_lift']:.2f}x")


Evidence receipts matched Week 6 and Week 7.
Final frame: 120,458 rows | 42 clients | 29 scoreable
Primary result: mean P@50 0.370 | mean client base rate 0.319
Wins: 22 of 29 | mean per-client lift 1.26x


## 2. Data

The source is build `flyrank_pseudonymized_warehouse_release_v20260703`, exported 3 July 2026. The warehouse spans 27 January 2025–30 June 2026 and contains 78,835,655 rows in `fact_content_daily_performance`; this study queried only selected January–March 2026 partitions and aggregated daily facts to page-level windows. In the saved March frame, features cover 1 February–1 March 2026 and the outcome covers 2–31 March; the earlier frame uses 1–29 January for features and 30 January–28 February for its outcome. The legacy `_prev30` name therefore describes the intended role, but the available prior window contains 29 calendar dates while the outcome contains 30.

The March analysis frame contains 120,507 content items with at least 10 prior-window impressions. The final four-feature complete-case frame contains 120,458 rows across 42 pseudonymous clients. Leave-one-client-out scoring was possible for 29 clients; 13 had fewer than 50 rows or a single outcome class and were reported as unscoreable rather than silently removed from the panel count.

The primary model uses `fact_content_daily_performance`. `dim_content` and `fact_content_query_90d` were inspected during the audit, but their freshness/query-derived fields were excluded from the final features because they were not reliably available before the outcome window. `dim_clients` was used for history-coverage context, never as a feature.

Public-safety exclusions include real client names, domains, raw URLs, titles, keywords, raw queries, and any lookup capable of reversing pseudonymous IDs. Pseudonymous client IDs are grouping keys only and content IDs never reach the paper.


In [2]:
data_receipt = pd.DataFrame([
    ("Release", "flyrank_pseudonymized_warehouse_release_v20260703", "Fixed pseudonymized snapshot"),
    ("Warehouse daily fact", "78,835,655 rows", "Release size; not the model-row count"),
    ("Warehouse date span", "2025-01-27 to 2026-06-30", "Unbalanced client histories"),
    ("Analysis partitions", "January-March 2026", "Selected windows only"),
    ("March aggregate", "120,507 pages", "At least 10 prior-window impressions"),
    ("Final complete frame", "120,458 pages / 42 clients", "Four decision-time features"),
    ("LOCO-scored subset", "29 clients", "At least 50 rows and both classes"),
], columns=["item", "value", "interpretation"])
print(data_receipt.to_string(index=False))


                item                                             value                        interpretation
             Release flyrank_pseudonymized_warehouse_release_v20260703          Fixed pseudonymized snapshot
Warehouse daily fact                                   78,835,655 rows Release size; not the model-row count
 Warehouse date span                          2025-01-27 to 2026-06-30           Unbalanced client histories
 Analysis partitions                                January-March 2026                 Selected windows only
     March aggregate                                     120,507 pages  At least 10 prior-window impressions
Final complete frame                        120,458 pages / 42 clients           Four decision-time features
  LOCO-scored subset                                        29 clients     At least 50 rows and both classes


## 3. Methodology

**Label.** `is_declining = imp_last30 < 0.8 × imp_prev30`: a page is positive when its outcome-window impressions are more than 20% below the prior window, after requiring at least 10 prior-window impressions. This is an observed traffic-change proxy for “worth reviewing,” not a diagnosis of content quality.

**Features.** The final random forest uses four measurements closed before the outcome window: prior-window impressions, clicks, average position (excluding zero-impression days), and number of active impression days. No outcome-window field, identifier, product score, post-window query statistic, or unreliable update date enters the model.

**Baseline.** The final null is random ordering within each held-out client; its expected Precision@50 is that client's observed positive rate on the same fold. The earlier hand-written rule—stale ≥180 days and visible ≥500 impressions—scored 0.100 against the random forest's 0.520 on the same exploratory seed-42 split, but it was retired after the timestamp audit.

**Validation.** The headline uses leave-one-client-out (LOCO) evaluation, training on every client except one and scoring that unseen client. Clients under 50 rows or with only one class are kept in the panel count but not assigned a P@50. A seven-seed grouped sweep measures sensitivity to which clients enter a test split, and a secondary February→March check tests forward-time behavior; that secondary check used the legacy query-filtered population and is therefore stress evidence, not the final headline.

**Leakage checks.** The audit tested label-derived fields, feature-window availability, population selection, and a positive control. Giving the model the label numerator produced P@50 = 1.000, proving the harness could detect leakage. Three original fields failed timing: `days_since_update` was negative for 80.1% of modeled rows, while `rare_share` and `anon_share` came from a fixed query window after the March outcome period; all three were removed, and the query-availability survival filter was removed with them.


In [3]:
feature_table = pd.DataFrame([
    ("imp_prev30", "sum of prior-window GSC impressions", "yes", "feature"),
    ("clk_prev30", "sum of prior-window GSC clicks", "yes", "feature"),
    ("pos_prev30", "mean position on prior-window impression days", "yes", "feature"),
    ("days_active_prev30", "prior-window days with impressions", "yes", "feature"),
    ("imp_last30 / is_declining", "outcome and label", "no", "evaluation only"),
    ("client/content hashes", "grouping and joins", "no", "context only"),
    ("days_since_update", "80.1% negative at the analysis anchor", "no", "timing failure"),
    ("rare_share / anon_share", "fixed query window after outcome", "no", "timing failure"),
], columns=["field", "definition", "model input", "status"])
print(feature_table.to_string(index=False))

validation_design = pd.DataFrame([
    ("Primary", "Leave one client out", "120,458-row honest frame", "Mean client P@50; own base rate"),
    ("Sensitivity", "7 grouped client splits", "legacy Week-5 frame", "Range across client draws"),
    ("Secondary", "February train -> March score", "legacy query-filtered frame", "Forward-time stress only"),
], columns=["role", "design", "population", "reported use"])
print("\n" + validation_design.to_string(index=False))


                    field                                    definition model input          status
               imp_prev30           sum of prior-window GSC impressions         yes         feature
               clk_prev30                sum of prior-window GSC clicks         yes         feature
               pos_prev30 mean position on prior-window impression days         yes         feature
       days_active_prev30            prior-window days with impressions         yes         feature
imp_last30 / is_declining                             outcome and label          no evaluation only
    client/content hashes                            grouping and joins          no    context only
        days_since_update         80.1% negative at the analysis anchor          no  timing failure
  rare_share / anon_share              fixed query window after outcome          no  timing failure

       role                        design                  population                    reported u

## 4. Results — model vs baseline on the same folds

The primary comparison uses the same 29 held-out-client folds. Mean model Precision@50 was **0.370**; the mean of those clients' observed base rates was **0.319**. The model beat the matched base rate for **22 of 29** clients, with **1.26× mean per-client lift**, but the range from **0.020 to 0.760** is as important as the average.

![Bar chart comparing mean held-out-client base rate of 0.319 with random-forest Precision at 50 of 0.370.](../figures/capstone-model-vs-baseline.svg)

**Takeaway:** the final gain is positive but modest in absolute terms; it is not the dramatic exploratory result.

![Anonymous held-out clients sorted by base rate, showing model Precision at 50 and matched base rate for each of 29 clients.](../figures/capstone-client-variability.svg)

**Takeaway:** the average hides substantial client heterogeneity, including seven clients where the ranker did not beat random ordering.

The seven-seed grouped split ranged from 0.280 to 0.580 (mean 0.489), showing that a single client draw is unstable. In the secondary February→March check, the four-feature model reached P@50 = 0.260 against a 0.174 base rate on the legacy filtered frame; when the test clients were also unseen, the mean was 0.171. Because that population inherited the later-discovered query-availability filter, these are stress-test numbers, not a substitute for a future, unfiltered holdout.

**Error analysis.** A retrospective, unfiltered top-50 queue contained 28 true declines and 22 misses (P@50 = 0.560 versus a 0.281 population base rate). The largest declines missed by the model had median prior-window impressions of 117,869, compared with 112.5 among surfaced true positives. The ranker can therefore miss high-impact pages even when volume is available; a human should never treat a low rank as permission to ignore a known business-critical page.


In [4]:
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "flyrank-capstone-mpl"))
import matplotlib.pyplot as plt

plt.switch_backend("Agg")
PAPER, INK, SLATE, SIGNAL, RULE = "#FBFAF7", "#14161A", "#2F4858", "#A8461F", "#D8D5CE"
plt.rcParams.update({
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER,
    "axes.edgecolor": RULE,
    "axes.labelcolor": INK,
    "xtick.color": SLATE,
    "ytick.color": SLATE,
    "font.family": "DejaVu Sans",
    "font.size": 10,
})

baseline = FINAL["mean_client_base_rate"]
model = FINAL["model_mean_p50"]
fig, ax = plt.subplots(figsize=(8.0, 4.3))
bars = ax.bar(["Random order\n(client base rate)", "Random forest\n(held-out client)"],
              [baseline, model], color=[SLATE, SIGNAL], width=0.55)
ax.set_ylim(0, 0.50)
ax.set_ylabel("Mean Precision@50")
ax.set_title("Final comparison on the same 29 LOCO folds", loc="left", fontsize=16, fontweight="bold", color=INK, pad=16)
ax.text(0.0, 1.01, "Each client is scored only after being held out of training", transform=ax.transAxes,
        color=SLATE, fontsize=9, va="bottom")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=RULE, linewidth=0.8, alpha=0.8)
ax.set_axisbelow(True)
for bar, value in zip(bars, [baseline, model]):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.012, f"{value:.3f}", ha="center", va="bottom",
            color=INK, fontsize=13, fontweight="bold")
fig.text(0.125, 0.01, "22 of 29 clients beat their own base rate; mean per-client lift 1.26x.", color=SLATE, fontsize=9)
fig.tight_layout(rect=[0, 0.05, 1, 1])
model_chart = FIGURE_DIR / "capstone-model-vs-baseline.svg"
fig.savefig(model_chart, format="svg", bbox_inches="tight", metadata={"Date": None})
fig.savefig(Path(tempfile.gettempdir()) / "capstone-model-vs-baseline.png", dpi=150, bbox_inches="tight")
plt.close(fig)

ordered = scoreable.sort_values("base_rate").reset_index(drop=True)
x = np.arange(1, len(ordered) + 1)
fig, ax = plt.subplots(figsize=(9.2, 4.7))
for xi, base_value, model_value in zip(x, ordered["base_rate"], ordered["p50"]):
    ax.plot([xi, xi], [base_value, model_value], color=RULE, linewidth=1.1, zorder=1)
ax.scatter(x, ordered["base_rate"], color=SLATE, marker="o", s=34, label="Matched client base rate", zorder=2)
ax.scatter(x, ordered["p50"], facecolors=PAPER, edgecolors=SIGNAL, marker="s", s=42,
           linewidths=1.8, label="Random-forest P@50", zorder=3)
ax.axhline(FINAL["model_median_p50"], color=SIGNAL, linestyle="--", linewidth=1,
           label=f"Model median {FINAL['model_median_p50']:.3f}")
ax.set_xlim(0, len(ordered) + 1)
ax.set_ylim(0, 0.84)
ax.set_xlabel("Anonymous held-out clients, sorted by base rate")
ax.set_ylabel("Precision@50 / base rate")
ax.set_title("Client variability is the result, not a footnote", loc="left", fontsize=16, fontweight="bold", color=INK, pad=16)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=RULE, linewidth=0.8, alpha=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=3, loc="upper left", fontsize=8.5)
fig.tight_layout()
variability_chart = FIGURE_DIR / "capstone-client-variability.svg"
fig.savefig(variability_chart, format="svg", bbox_inches="tight", metadata={"Date": None})
fig.savefig(Path(tempfile.gettempdir()) / "capstone-client-variability.png", dpi=150, bbox_inches="tight")
plt.close(fig)

for source in (model_chart, variability_chart):
    (DOC_ASSET_DIR / source.name).write_bytes(source.read_bytes())

honest_comparison = pd.DataFrame([
    ("Random ordering", baseline, "mean observed base rate across the same 29 held-out clients"),
    ("Random forest", model, "mean LOCO P@50 across the same 29 held-out clients"),
], columns=["method", "mean_p50", "scope"])
print(honest_comparison.to_string(index=False, formatters={"mean_p50": "{:.3f}".format}))

audit_trace = pd.DataFrame([
    ("Week-5 seven features + query-filtered population", 0.489, 0.185, 0.304),
    ("Four decision-time features + same population", 0.329, 0.185, 0.144),
    ("Four decision-time features + unfiltered population", 0.354, 0.343, 0.011),
], columns=["seven-seed grouped comparison", "mean_p50", "mean_test_base", "absolute_lift"])
print("\nAudit trail (same seven grouped seeds within each row; not the LOCO headline):")
print(audit_trace.to_string(index=False))
print(f"\nWrote figures: {model_chart.relative_to(PROJECT_ROOT)}, {variability_chart.relative_to(PROJECT_ROOT)}")


         method mean_p50                                                       scope
Random ordering    0.319 mean observed base rate across the same 29 held-out clients
  Random forest    0.370          mean LOCO P@50 across the same 29 held-out clients

Audit trail (same seven grouped seeds within each row; not the LOCO headline):
                      seven-seed grouped comparison  mean_p50  mean_test_base  absolute_lift
  Week-5 seven features + query-filtered population     0.489           0.185          0.304
      Four decision-time features + same population     0.329           0.185          0.144
Four decision-time features + unfiltered population     0.354           0.343          0.011

Wrote figures: work\figures\capstone-model-vs-baseline.svg, work\figures\capstone-client-variability.svg


## 5. Limitations and honest framing

- **One panel and short evaluation period.** Results are observed in selected January–March 2026 partitions of an unbalanced client panel; they do not establish seasonality or long-run stability.
- **A threshold proxy.** A 20% impression drop is a practical label, not a diagnosis. Pages just above and below the threshold may be operationally similar.
- **Unequal saved window lengths.** The selected-partition SQL gives the prior frame 29 available dates and the outcome 30, despite the legacy `_prev30` name. This may slightly affect the decline label and should be corrected in a rerun that includes the missing boundary date.
- **Uneven generalization.** P@50 ranges from 0.020 to 0.760, seven scoreable clients do not beat their base rate, and 13 clients cannot be scored at all under the stated minimum.
- **Forward-time evidence is incomplete.** The saved February→March check used a population later found to be query-availability filtered. A future unfiltered temporal holdout remains required before any production proposal.
- **No causal effect.** The study did not randomize refresh actions and cannot claim that editing a flagged page causes recovery.
- **No semantic diagnosis.** The public release contains pseudonymized metrics, not page text or private queries. The model cannot know factual quality, intent fit, brand constraints, or legal risk.
- **Action weights are planning heuristics.** The Week-7 50/30/20 re-rank is not a calibrated probability, revenue forecast, treatment-effect estimate, or cross-client score.

The defensible claim is deliberately narrow: this model sometimes concentrates observed declines in a human review queue better than random ordering for held-out clients in this panel. It does not predict Google's algorithm, promise recovery, or authorize a content change.


In [5]:
limitations = {
    "causal_effect_estimated": False,
    "future_unfiltered_holdout_completed": False,
    "cross_client_calibration_claimed": False,
    "small_or_single_class_clients_scored": False,
    "semantic_content_quality_observed": False,
    "automatic_action_authorized": False,
}
print(pd.Series(limitations, name="claimed").to_string())
assert not any(limitations.values())


causal_effect_estimated                 False
future_unfiltered_holdout_completed     False
cross_client_calibration_claimed        False
small_or_single_class_clients_scored    False
semantic_content_quality_observed       False
automatic_action_authorized             False


## 6. Ranked recommendations — the action playbook

The model score is not the action. Week 7 consumes the model review score available in the regenerated queue interface and combines it with a within-client opportunity proxy and evidence quality using planning weights of 50% / 30% / 20%, then maps each row to an archetype, action, reason codes, review band, and a mandatory human gate. The executed receipt below records the interface scope. Those weights organize work; they were not fitted against revenue or treatment outcomes, and the Week-6 validation metrics do not transfer to this heuristic re-rank.

**Review sequence after queue sorting:**

1. **Diagnose decline-risk pages.** Check tracking, indexation, seasonality, intent shifts, competitors, and cannibalization before proposing an edit.
2. **Protect visible winners.** High visibility raises the cost of an unnecessary change; diagnose before touching a page-one result.
3. **Review snippet and intent for CTR opportunities.** Compare query intent and the live SERP; low aggregate CTR alone does not prove a title problem.
4. **Review content experience for engagement gaps.** Verify analytics coverage, structure, intent fit, and page experience before attributing the signal to content.
5. **Refresh only after verifying decay.** Staleness alone is not permission to edit. The audit found the original freshness field unavailable for 80.1% of rows at the analysis anchor, so update dates must be verified and factual/intent decay confirmed manually.
6. **Monitor thin or limited-signal pages.** Do not add words to hit a count and do not act until enough evidence exists.

**Human-review rule:** every row requires an accept / defer / reject decision with a reason. The system may rank pseudonymous rows and attach reason codes; it may not rewrite, publish, redirect, delete, canonicalize, no-index, diagnose causation, promise recovery, compare scores across clients, retrain, change thresholds, or deploy automatically.

**Monitoring / retrain gates:** stop on contract failures; inspect missingness shifts above 5 percentage points; audit score/action drift at PSI > 0.20 or action-share movement > 0.20; pause a client's ranking after two mature windows at or below its base rate; inspect repeated reviewer rejection; keep new/small clients in shadow mode. Every trigger opens a human review—none starts automatic retraining.


In [6]:
action_policy = pd.DataFrame([
    (1, "decline_risk_review", "manual_diagnostic_review", "model_ranked_candidate | model_decline_risk", 20),
    (2, "protect_winner", "diagnose_before_editing", "visible_demand | page_one_to_protect", 20),
    (3, "ctr_opportunity", "review_serp_snippet_and_intent", "visible_demand | low_ctr_visible_page", 15),
    (4, "engagement_gap", "review_content_experience", "visible_demand | weak_engagement_signal", 25),
    (5, "stale_visible", "manual_refresh_review", "visible_demand | verified_staleness", 30),
    (6, "thin_visible", "review_depth_and_intent", "visible_demand | thin_content_signal", 35),
    (7, "limited_signal", "monitor_and_collect_data", "limited_evidence", 5),
], columns=["rank", "archetype", "recommended_action", "example_reason_codes", "assumed_minutes"])
print(action_policy.to_string(index=False))

print(f"\nExecuted Week-7 action source: {w07['source_mode']}")
print(f"Rows / clients: {w07['rows_exported']:,} / {w07['clients']}")
print("Action counts are descriptive of that input, not model-validation results:")
print(pd.Series(w07["action_counts"], name="rows").sort_values(ascending=False).to_string())
print(f"\nReview-now capacity illustration: {w07['review_band_counts'].get('review_now', 0)} rows, "
      f"{w07['review_now_planning_minutes']:,} assumed minutes.")
print(w07["planning_minutes_status"])

triggers = pd.DataFrame(w07["monitoring_triggers"])
print("\nMonitoring gates:")
print(triggers.to_string(index=False))
assert not triggers["automatic_retrain"].any()


 rank           archetype             recommended_action                        example_reason_codes  assumed_minutes
    1 decline_risk_review       manual_diagnostic_review model_ranked_candidate | model_decline_risk               20
    2      protect_winner        diagnose_before_editing        visible_demand | page_one_to_protect               20
    3     ctr_opportunity review_serp_snippet_and_intent       visible_demand | low_ctr_visible_page               15
    4      engagement_gap      review_content_experience     visible_demand | weak_engagement_signal               25
    5       stale_visible          manual_refresh_review         visible_demand | verified_staleness               30
    6        thin_visible        review_depth_and_intent        visible_demand | thin_content_signal               35
    7      limited_signal       monitor_and_collect_data                            limited_evidence                5

Executed Week-7 action source: full_regenerated_queue
R

## 7. Reproducibility

The repository is the executable record: [GitHub repository](https://github.com/muneeb-khokhar/flyrank-ml-track), [data contract](https://github.com/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb), [model](https://github.com/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w05_model.ipynb), [validation audit](https://github.com/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w06_validation_audit.ipynb), [action playbook](https://github.com/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w07_action_playbook.ipynb), and [this capstone notebook](https://github.com/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/capstone.ipynb).

From a fresh clone:

```bash
python -m pip install -r requirements.txt
python scripts/run_all.py
jupyter lab
```

Then run `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`, and `capstone.ipynb` top to bottom. Full-warehouse reproduction requires approved Hugging Face access and a local `hf-token.txt` as documented in `SETUP.md`; never commit that token. The random forest uses 300 trees with `random_state=42`; grouped sensitivity seeds are `0, 1, 7, 13, 42, 99, 2024`.

The committed receipts are `work/outputs/w07_playbook_metrics.json` and `work/outputs/capstone_metrics.json`. Row-level queue CSVs remain ignored and are regenerated locally by design.


In [7]:
capstone_metrics = {
    "artifact": "capstone_research_paper_receipt",
    "author": "Muneeb Ur Rehman",
    "release": {
        "build_id": "flyrank_pseudonymized_warehouse_release_v20260703",
        "export_date": "2026-07-03",
        "daily_fact_rows": 78_835_655,
        "daily_fact_date_min": "2025-01-27",
        "daily_fact_date_max": "2026-06-30",
    },
    "analysis": {
        "partitions": ["2026-01", "2026-02", "2026-03"],
        "march_feature_window": ["2026-02-01", "2026-03-01"],
        "march_outcome_window": ["2026-03-02", "2026-03-31"],
        "window_length_caveat": "saved prior window has 29 available calendar dates; outcome has 30",
        "extracted_pages": FINAL["extracted_pages"],
        "honest_complete_rows": FINAL["honest_complete_rows"],
        "honest_total_clients": FINAL["honest_total_clients"],
        "scoreable_clients": FINAL["scoreable_clients"],
        "unscoreable_clients": FINAL["unscoreable_clients"],
        "label": "imp_last30 < 0.8 * imp_prev30; imp_prev30 >= 10",
        "features": validation["honest_features"],
    },
    "primary_result": {
        "design": "leave_one_client_out",
        "metric": "precision_at_50",
        "model_mean": round(FINAL["model_mean_p50"], 3),
        "model_median": round(FINAL["model_median_p50"], 3),
        "model_range": [round(FINAL["model_min_p50"], 3), round(FINAL["model_max_p50"], 3)],
        "matched_mean_client_base_rate": round(FINAL["mean_client_base_rate"], 3),
        "clients_beating_own_base_rate": FINAL["clients_beating_base"],
        "scoreable_clients": FINAL["scoreable_clients"],
        "mean_per_client_lift": round(FINAL["mean_per_client_lift"], 2),
    },
    "secondary_checks": {
        "grouped_seed_sweep_p50_range": [0.280, 0.580],
        "grouped_seed_sweep_p50_mean": 0.489,
        "time_aware_legacy_frame_p50": 0.260,
        "time_aware_legacy_frame_base_rate": 0.174,
        "time_aware_unseen_client_mean_p50": 0.171,
        "retrospective_unfiltered_top50_p50": 0.560,
        "retrospective_unfiltered_population_base_rate": 0.281,
    },
    "action_layer": {
        "source_mode": w07["source_mode"],
        "rows": w07["rows_exported"],
        "clients": w07["clients"],
        "priority_formula": w07["priority_formula"],
        "not_claimed": w07["not_claimed"],
        "no_go": w07["no_go"],
    },
    "source_receipts": {
        "w06_path": str(W06_PATH.relative_to(PROJECT_ROOT)),
        "w06_sha256": hashlib.sha256(W06_PATH.read_bytes()).hexdigest(),
        "w07_path": str(W07_PATH.relative_to(PROJECT_ROOT)),
        "w07_sha256": hashlib.sha256(W07_PATH.read_bytes()).hexdigest(),
    },
    "paper_status": "GitHub Pages-ready; live deployment must be verified by the author after push",
}
CAPSTONE_METRICS = OUTPUT_DIR / "capstone_metrics.json"
CAPSTONE_METRICS.write_text(json.dumps(capstone_metrics, indent=2, sort_keys=True), encoding="utf-8")

# Keep the two action-capacity statements on the static paper synchronized with the executed Week-7 receipt.
paper_path = PROJECT_ROOT / "docs/index.html"
paper_html = paper_path.read_text(encoding="utf-8")
bands = w07["review_band_counts"]
scope_block = (
    '<!-- ACTION_SCOPE_START -->\n'
    '          <div class="definition">\n'
    f'            <p><strong>Executed action-interface scope.</strong> The local pipeline regenerated {w07["rows_exported"]:,} pseudonymous rows across {w07["clients"]} clients: '
    f'{bands.get("review_now", 0):,} <code>review_now</code>, {bands.get("review_next", 0):,} <code>review_next</code>, and '
    f'{bands.get("monitor_later", 0):,} <code>monitor_later</code>. These counts describe the action input, not model-validation performance, and the row-level CSV remains out of Git.</p>\n'
    '          </div>\n'
    '          <!-- ACTION_SCOPE_END -->'
)
capacity_block = (
    '<!-- ACTION_CAPACITY_START -->\n'
    f'          <p>The action layer assigns five to thirty-five assumed minutes by archetype to turn a review band into a capacity estimate. In this run, the {bands.get("review_now", 0):,} <code>review_now</code> rows total {w07["review_now_planning_minutes"]:,} assumed minutes (about {w07["review_now_planning_minutes"] / 60:.1f} hours). That estimate is for staffing a review cycle only. No expected traffic or revenue uplift is calculated because the study has no treatment-effect data. High-visibility winners receive protective review because the downside of an unnecessary edit is greater; limited-signal pages are deferred because acting on noise spends time without evidence.</p>\n'
    '          <!-- ACTION_CAPACITY_END -->'
)
paper_html, scope_replacements = re.subn(
    r'<!-- ACTION_SCOPE_START -->.*?<!-- ACTION_SCOPE_END -->', scope_block, paper_html, flags=re.S
)
paper_html, capacity_replacements = re.subn(
    r'<!-- ACTION_CAPACITY_START -->.*?<!-- ACTION_CAPACITY_END -->', capacity_block, paper_html, flags=re.S
)
assert scope_replacements == 1 and capacity_replacements == 1, "Static-paper action markers are missing or duplicated."
paper_path.write_text(paper_html, encoding="utf-8")

print(f"Wrote receipt: {CAPSTONE_METRICS.relative_to(PROJECT_ROOT)} ({CAPSTONE_METRICS.stat().st_size:,} bytes)")
print(f"Synchronized action scope in: {paper_path.relative_to(PROJECT_ROOT)}")
print(json.dumps(capstone_metrics["primary_result"], indent=2))


Wrote receipt: work\outputs\capstone_metrics.json (3,311 bytes)
Synchronized action scope in: docs\index.html
{
  "design": "leave_one_client_out",
  "metric": "precision_at_50",
  "model_mean": 0.37,
  "model_median": 0.3,
  "model_range": [
    0.02,
    0.76
  ],
  "matched_mean_client_base_rate": 0.319,
  "clients_beating_own_base_rate": 22,
  "scoreable_clients": 29,
  "mean_per_client_lift": 1.26
}


## 8. Artifacts the paper embeds + page-format research

The deployed artifact uses a single long-form page with an abstract first, an in-page table of contents, narrow reading measure, visible metric definitions, matched-split comparison, descriptive captions, and the limitations beside—not hidden behind—the result. Charts have a plain-language takeaway in adjacent text, tables have semantic headers and scroll within their own container on narrow screens, and the page reflows to one column.

This choice follows current primary guidance: [W3C headings and labels](https://www.w3.org/WAI/WCAG22/Understanding/headings-and-labels.html), [W3C minimum contrast](https://www.w3.org/WAI/WCAG22/Understanding/contrast-minimum), [W3C reflow](https://www.w3.org/WAI/WCAG22/Understanding/reflow.html), [GOV.UK content design](https://www.gov.uk/guidance/content-design/what-is-content-design), and [GitHub's official Pages publishing-source documentation](https://docs.github.com/en/pages/getting-started-with-github-pages/configuring-a-publishing-source-for-your-github-pages-site).

Generated paper artifacts:

- `work/figures/capstone-model-vs-baseline.svg`
- `work/figures/capstone-client-variability.svg`
- `work/outputs/capstone_metrics.json`
- `docs/index.html` and `docs/assets/`


In [8]:
artifacts = [
    FIGURE_DIR / "capstone-model-vs-baseline.svg",
    FIGURE_DIR / "capstone-client-variability.svg",
    OUTPUT_DIR / "capstone_metrics.json",
    PROJECT_ROOT / "docs/index.html",
    DOC_ASSET_DIR / "capstone-model-vs-baseline.svg",
    DOC_ASSET_DIR / "capstone-client-variability.svg",
]
for artifact in artifacts:
    status = "OK" if artifact.exists() else "MISSING"
    size = artifact.stat().st_size if artifact.exists() else 0
    print(f"{status:7} {artifact.relative_to(PROJECT_ROOT)} ({size:,} bytes)")


OK      work\figures\capstone-model-vs-baseline.svg (57,800 bytes)
OK      work\figures\capstone-client-variability.svg (64,631 bytes)
OK      work\outputs\capstone_metrics.json (3,311 bytes)
OK      docs\index.html (45,820 bytes)
OK      docs\assets\capstone-model-vs-baseline.svg (57,800 bytes)
OK      docs\assets\capstone-client-variability.svg (64,631 bytes)


## 9. Acknowledgments & data credit

[Built on the FlyRank ML Internship dataset](https://flyrank.ai). I also acknowledge the internship's research and data-safety framework, which made the pseudonymized release and its documentation available for this study.


## Self-check

- [x] Title and five-sentence abstract appear first.
- [x] Introduction states the decision, unit, action, and error costs.
- [x] Data names the release, tables, date windows, exclusions, and actual modeled row count.
- [x] Methodology states assumptions, four features, label, baseline, validation design, and leakage checks.
- [x] Results compare model and baseline on the same folds and include two charts with takeaways.
- [x] Limitations use observed / measured / directional / decision-support framing.
- [x] Recommendations include reason codes, archetype mapping, refresh/decay insight, cost assumptions, human review, monitoring, retrain gates, and no-go cases.
- [x] Reproducibility links notebooks and names commands, seeds, receipts, and token handling.
- [x] Acknowledgments carry the required linked FlyRank data credit.
- [x] The page source is public-safe and GitHub Pages-ready.
- [x] The closing section contains a timed five-minute demo outline.
- [x] The social post and three-sentence employer summary are honest and shareable as written.


In [9]:
paper_path = PROJECT_ROOT / "docs/index.html"
paper = paper_path.read_text(encoding="utf-8")
checks = {
    "paper_exists": paper_path.exists(),
    "abstract_present": 'id="abstract"' in paper,
    "all_required_sections": all(
        f'id="{section}"' in paper
        for section in ["introduction", "data", "methodology", "results", "limitations", "recommendations", "reproducibility", "acknowledgments"]
    ),
    "flyrank_credit_linked": 'href="https://flyrank.ai"' in paper and "Built on the FlyRank ML Internship dataset" in paper,
    "figures_embedded": all(name in paper for name in ["capstone-model-vs-baseline.svg", "capstone-client-variability.svg"]),
    "no_pseudonymous_row_ids": not re.search(r"(?:client|content)_[0-9a-f]{8,}", paper, flags=re.I),
    "no_private_contact_material": "muneebkhokher38" not in paper.lower(),
    "metrics_receipt_exists": CAPSTONE_METRICS.exists(),
    "action_queue_requires_human": (
        "human review" in w07["intended_use"]
        and any("no automatic rewrite" in item for item in w07["no_go"])
        and any("no automatic retrain" in item for item in w07["no_go"])
    ),
    "no_automatic_retrain": not any(item["automatic_retrain"] for item in w07["monitoring_triggers"]),
}
failed = [name for name, passed in checks.items() if not passed]
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert not failed, f"Capstone self-check failed: {failed}"

submission_value = (PROJECT_ROOT / "submission/paper_url.txt").read_text(encoding="utf-8").strip()
if submission_value.startswith("https://"):
    print(f"\nSubmission URL recorded: {submission_value}")
else:
    print("\nDeployment is not claimed yet. After GitHub Pages returns HTTP 200, replace the placeholder")
    print("with the exact live URL on one line, then run this cell once more.")
print(f"\nAll {len(checks)} content, safety, and artifact checks passed.")


PASS  paper_exists
PASS  abstract_present
PASS  all_required_sections
PASS  flyrank_credit_linked
PASS  figures_embedded
PASS  no_pseudonymous_row_ids
PASS  no_private_contact_material
PASS  metrics_receipt_exists
PASS  action_queue_requires_human
PASS  no_automatic_retrain

Submission URL recorded: https://muneeb-khokhar.github.io/flyrank-ml-track/

All 10 content, safety, and artifact checks passed.


## 10. Five-minute demo outline

**0:00–0:40 — Question and FlyRank problem**  
FlyRank's Refresh / Content Opportunity Scoring lane asks a practical question: when search performance declines across more pages than a content team can inspect, which pages should an editor review first? My goal was a public-safe, human-reviewed priority queue—not automatic editing.

**0:40–1:30 — Data and method**  
I aggregated selected January–March 2026 partitions from FlyRank's pseudonymized 78,835,655-row search-performance release into a 120,507-page March frame. A random forest used four prior-window features—impressions, clicks, average position, and active days—and I tested it with leave-one-client-out Precision@50 against each held-out client's base rate. I excluded identifiers, outcome-window fields, and three features that failed timing or population-leakage checks.

**1:30–2:30 — One chart**  
Show `work/figures/capstone-model-vs-baseline.svg`. Explain that both bars use the same 29 held-out-client folds: model mean Precision@50 is 0.370 and the matched mean client base rate is 0.319.

**2:30–3:35 — One honest result**  
The model beat the matched base rate for 22 of 29 scoreable clients, but performance ranged from 0.020 to 0.760 and 13 additional clients were too small or single-class to score. The gain is useful but modest and uneven; it does not show that refreshing a flagged page will cause recovery.

**3:35–4:35 — One recommendation**  
Use the score only to order a review queue. For a high-ranked decline-risk page, an editor should first verify tracking, indexation, seasonality, intent shifts, competitors, and cannibalization, then accept, defer, or reject the proposed review action with a reason. No rewriting, publishing, redirecting, deleting, threshold changes, or retraining should happen automatically.

**4:35–5:00 — Close**  
The contribution is not an autonomous content system. It is a reproducible decision-support workflow that leaves its leakage checks, limits, reason codes, and human gate visible.


## 11. Shareable cuts

### Short social post

How do you rank content pages for human review without turning a model score into an automatic edit? I built a leakage-audited review queue for FlyRank's Refresh / Content Opportunity Scoring lane using a 120,507-page aggregate derived from a pseudonymized 78.8M-row search-performance release. A four-feature random forest was tested with leave-one-client-out validation against each held-out client's base rate: mean Precision@50 was 0.370 versus 0.319, and it beat the matched base rate for 22 of 29 scoreable clients. The result is modest and uneven, so the deliverable is a reason-coded, human-reviewed queue—not an autonomous publishing system. Read the paper: https://muneeb-khokhar.github.io/flyrank-ml-track/

### Employer-facing summary — three sentences

I built a reproducible, leakage-audited machine-learning pipeline and reason-coded review queue that helps prioritize content pages for human editorial review. It uses a 120,507-page analysis frame aggregated from FlyRank's pseudonymized 78,835,655-row search-performance release, with time-safe features and leave-one-client-out validation. The random forest achieved mean Precision@50 of 0.370 versus a matched 0.319 client base rate and outperformed that baseline for 22 of 29 scoreable clients, supporting cautious human prioritization rather than automated edits.
